Install Kaggle data ~ 4.06 gb

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d nih-chest-xrays/sample

Dataset URL: https://www.kaggle.com/datasets/nih-chest-xrays/sample
License(s): CC0-1.0
sample.zip: Skipping, found more recently modified local copy (use --force to force download)


Import labels ~ Convert to data table (Patient id, Age, Gender, Orginal Image)

In [ ]:
import pandas as pd

labels = pd.read_csv("sample/sample_labels.csv")

print("Dataset loaded")
labels.head()

Dataset loaded


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImageWidth,OriginalImageHeight,OriginalImagePixelSpacing_x,OriginalImagePixelSpacing_y
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,5,13,060Y,M,AP,3056,2544,0.139,0.139
1,00000013_026.png,Cardiomegaly|Emphysema,26,13,057Y,M,AP,2500,2048,0.168,0.168
2,00000017_001.png,No Finding,1,17,077Y,M,AP,2500,2048,0.168,0.168
3,00000030_001.png,Atelectasis,1,30,079Y,M,PA,2992,2991,0.143,0.143
4,00000032_001.png,Cardiomegaly|Edema|Effusion,1,32,055Y,F,AP,2500,2048,0.168,0.168


In [ ]:
# Create cancer proxy label
labels['cancer'] = labels['Finding Labels'].apply(
    lambda x: 1 if ('Mass' in x or 'Nodule' in x) else 0
)

labels[['Image Index', 'Finding Labels', 'cancer']].head()

,Image Index,Finding Labels,cancer
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,0
1,00000013_026.png,Cardiomegaly|Emphysema,0
2,00000017_001.png,No Finding,0
3,00000030_001.png,Atelectasis,0
4,00000032_001.png,Cardiomegaly|Edema|Effusion,0


In [ ]:
labels['cancer'].value_counts()

,count
cancer,
0,5053
1,553


In [ ]:
!unzip sample.zip


Archive:  sample.zip
replace sample/images/00000013_005.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls sample/sample

images


In [ ]:
!ls sample

images	sample	sample_labels.csv


In [ ]:
import pandas as pd

labels = pd.read_csv("sample/sample_labels.csv")

labels.head()

,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImageWidth,OriginalImageHeight,OriginalImagePixelSpacing_x,OriginalImagePixelSpacing_y
0,00000013_005.png,Emphysema|Infiltration|Pleural_Thickening|Pneu...,5,13,060Y,M,AP,3056,2544,0.139,0.139
1,00000013_026.png,Cardiomegaly|Emphysema,26,13,057Y,M,AP,2500,2048,0.168,0.168
2,00000017_001.png,No Finding,1,17,077Y,M,AP,2500,2048,0.168,0.168
3,00000030_001.png,Atelectasis,1,30,079Y,M,PA,2992,2991,0.143,0.143
4,00000032_001.png,Cardiomegaly|Edema|Effusion,1,32,055Y,F,AP,2500,2048,0.168,0.168


In [ ]:
labels['cancer'] = labels['Finding Labels'].apply(
    lambda x: 1 if ('Mass' in x or 'Nodule' in x) else 0
)

In [ ]:
val_data = datagen.flow_from_dataframe(
    dataframe=labels,
    directory="sample/sample/images",
    x_col="Image Index",
    y_col="cancer",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="validation"
)

Found 1121 validated image filenames.


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_dataframe(
    dataframe=labels,
    directory="sample/sample/images",
    x_col="Image Index",
    y_col="cancer",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="training",
    shuffle=True
)

val_data = datagen.flow_from_dataframe(
    dataframe=labels,
    directory="sample/sample/images",
    x_col="Image Index",
    y_col="cancer",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="raw",
    subset="validation"
)

Found 4485 validated image filenames.
Found 1121 validated image filenames.


Iport DenseNet121 (Cnn Model) ~ From tensorflow pretrianed

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,038,529 (26.85 MB)

 Trainable params: 6,954,881 (26.53 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 448s 942ms/step - accuracy: 0.8946 - auc: 0.5207 - loss: 0.3619 - val_accuracy: 0.9135 - val_auc: 0.5215 - val_loss: 0.5259
Epoch 2/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 102s 364ms/step - accuracy: 0.8943 - auc: 0.5000 - loss: 0.3471 - val_accuracy: 0.8956 - val_auc: 0.4701 - val_loss: 0.4802
Epoch 3/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 102s 363ms/step - accuracy: 0.8938 - auc: 0.4824 - loss: 0.3500 - val_accuracy: 0.9242 - val_auc: 0.4914 - val_loss: 0.4742
Epoch 4/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 104s 371ms/step - accuracy: 0.8906 - auc: 0.4834 - loss: 0.3523 - val_accuracy: 0.9242 - val_auc: 0.5324 - val_loss: 0.3520
Epoch 5/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 103s 367ms/step - accuracy: 0.8953 - auc: 0.4861 - loss: 0.3448 - val_accuracy: 0.9242 - val_auc: 0.5307 - val_loss: 0.3943


In [ ]:
from sklearn.utils import class_weight
import numpy as np

weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(labels['cancer']),
    y=labels['cancer']
)

class_weights = {0: weights[0], 1: weights[1]}
print(class_weights)

{0: np.float64(0.5547199683356422), 1: np.float64(5.06871609403255)}


In [ ]:
model.save("lung_cancer_densenet_model.h5")

NameError: name 'model' is not defined

In [ ]:
import numpy as np

# Get predictions
pred_probs = model.predict(val_data)

# Convert probabilities to 0/1 labels
pred_labels = (pred_probs > 0.5).astype(int)

# True labels
true_labels = val_data.labels

26/71 ━━━━━━━━━━━━━━━━━━━━ 12s 275ms/step

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(true_labels, pred_labels)
precision = precision_score(true_labels, pred_labels)
recall = recall_score(true_labels, pred_labels)
f1 = f1_score(true_labels, pred_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precision_vals, recall_vals, thresholds = precision_recall_curve(true_labels, pred_probs)

plt.figure(figsize=(6,5))
plt.plot(recall_vals, precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
# Extract clinical features

labels['age'] = labels['Patient Age'].str.replace('[^0-9]', '', regex=True).astype(int)

labels['gender'] = labels['Patient Gender'].map({'M':1, 'F':0})

labels[['age','gender']].head()

,age,gender
0,60,1
1,57,1
2,77,1
3,79,1
4,55,0


In [ ]:
from tensorflow.keras.layers import Input, Dense, Concatenate, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import DenseNet121

In [ ]:
# Image input
image_input = Input(shape=(224,224,3))

base_model = DenseNet121(
    weights='imagenet',
    include_top=False,
    input_tensor=image_input
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

# Clinical input (Age + Gender)
clinical_input = Input(shape=(2,))
clinical_branch = Dense(16, activation='relu')(clinical_input)

# Combine both feature sets
combined = Concatenate()([x, clinical_branch])

z = Dense(64, activation='relu')(combined)
output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[image_input, clinical_input], outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_2    │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d_2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_3    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_3… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,104,241 (27.10 MB)

 Trainable params: 7,020,593 (26.78 MB)

 Non-trainable params: 83,648 (326.75 KB)

In [ ]:
def multimodal_generator(image_gen, labels):
    while True:
        img_batch, y_batch = next(image_gen)

        # Skip incomplete batch
        if img_batch.shape[0] != image_gen.batch_size:
            continue

        batch_files = image_gen.filenames[
            image_gen.batch_index * image_gen.batch_size :
            image_gen.batch_index * image_gen.batch_size + img_batch.shape[0]
        ]

        batch_files = [f.split('/')[-1] for f in batch_files]

        clinical = labels.set_index('Image Index').loc[batch_files][['age','gender']].values.astype('float32')

        yield (img_batch, clinical), y_batch

In [ ]:
train_gen = multimodal_generator(train_data, labels)
val_gen = multimodal_generator(val_data, labels)

In [ ]:
history = model.fit(
    train_gen,
    steps_per_epoch=len(train_data),
    validation_data=val_gen,
    validation_steps=len(val_data),
    epochs=5
)

Epoch 1/5
279/281 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.8743 - loss: 0.4339

InvalidArgumentError: Graph execution error:

Detected at node functional_1_1/concatenate_1/concat defined at (most recent call last):
<stack traces unavailable>
Cannot concatenate arrays that differ in dimensions other than the one being concatenated. Dimension 0 in both shapes must be equal (or compatible): f32[16,1024] vs f32[5,16].
	 [[{{node functional_1_1/concatenate_1/concat}}]]
	tf2xla conversion failed while converting __inference_one_step_on_data_210220[]. Run with TF_DUMP_GRAPH_PREFIX=/path/to/dump/dir and --vmodule=xla_compiler=2 to obtain a dump of the compiled functions.
	 [[StatefulPartitionedCall]] [Op:__inference_multi_step_on_iterator_212929]

In [ ]:
import pandas as pd
import numpy as np

labels = pd.read_csv("sample/sample_labels.csv")

# cancer proxy label
labels["cancer"] = labels["Finding Labels"].apply(lambda x: 1 if ("Mass" in x or "Nodule" in x) else 0)

# age + gender
labels["age"] = labels["Patient Age"].str.extract(r"(\d+)").astype(int)
labels["gender"] = labels["Patient Gender"].map({"M": 1, "F": 0}).astype(int)

labels.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(labels, test_size=0.15, random_state=42, stratify=labels["cancer"])

In [ ]:
import tensorflow as tf

IMG_SIZE = 224
BATCH_SIZE = 16
IMG_DIR = "sample/sample/images"

def preprocess_row(image_name, age, gender, y):
    # image
    path = tf.strings.join([IMG_DIR, "/", image_name])
    img = tf.io.read_file(path)
    img = tf.image.decode_png(img, channels=3)          # images are png in this sample
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0

    # clinical features (normalize age a bit so it’s not huge)
    age = tf.cast(age, tf.float32) / 100.0
    gender = tf.cast(gender, tf.float32)
    clin = tf.stack([age, gender], axis=0)              # shape (2,)

    y = tf.cast(y, tf.float32)
    return (img, clin), y

def make_dataset(df, training=True):
    ds = tf.data.Dataset.from_tensor_slices((
        df["Image Index"].values,
        df["age"].values,
        df["gender"].values,
        df["cancer"].values
    ))
    ds = ds.map(preprocess_row, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2000)
    ds = ds.batch(BATCH_SIZE, drop_remainder=True)      # prevents last-batch shape issues
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Concatenate
from tensorflow.keras.models import Model

image_input = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="xray")
clinical_input = Input(shape=(2,), name="clinical")  # [age_norm, gender]

base = DenseNet121(weights="imagenet", include_top=False, input_tensor=image_input)
x = GlobalAveragePooling2D()(base.output)

c = Dense(16, activation="relu")(clinical_input)

combined = Concatenate()([x, c])
z = Dense(64, activation="relu")(combined)
out = Dense(1, activation="sigmoid")(z)

model = Model(inputs=[image_input, clinical_input], outputs=out)
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])

model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)

In [ ]:
import matplotlib.pyplot as plt

# Plot training history
plt.figure(figsize=(12, 5))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Last convolutional layer for DenseNet121
LAST_CONV_LAYER = "conv5_block16_concat"

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):

    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)

    return heatmap.numpy()

In [ ]:
# Load image
img = cv2.imread(img_path)
img = cv2.resize(img, (224,224))


corners = cv2.goodFeaturesToTrack(
    gray1, maxCorners=200, qualityLevel=0.01, minDistance=20
)


corners =
img_array = np.expand_dims(img / 255.0, axis=0)

# Generate heatmap
heatmap = make_gradcam_heatmap(
    img_array,
    model,
    LAST_CONV_LAYER
)

# Resize heatmap
heatmap = cv2.resize(heatmap, (224,224))
heatmap = np.uint8(255 * heatmap)

# Apply colormap
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

# Overlay
superimposed_img = heatmap * 0.4 + img

# Show result
plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(np.uint8(superimposed_img), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title("Grad-CAM Heatmap")
plt.show()

NameError: name 'cv2' is not defined